In [1]:
import asyncio
import time
import os
import json
import hashlib
from typing import List, Dict, Tuple
from urllib.parse import urlparse
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig
from crawl4ai.deep_crawling import BFSDeepCrawlStrategy
from bs4 import BeautifulSoup
from firecrawl import FirecrawlApp
from google.oauth2 import service_account
from googleapiclient.discovery import build


CATEGORY_THRESHOLDS = {
		"ABOUT_US": 200,
		"EBOOK": 200,
		"COURSES": 300,
		"RECENT_BLOG": 450,
		"TESTIMONIALS": 100,
		"WEBINAR": 150,
		"SERVICES": 150,
		"PODCAST": 200,
		"SHOP": 100,
}

CATEGORY_KEYWORDS = {
		"ABOUT_US": ["about", "who-we-are", "company", "our-story"],
		"EBOOK": ["ebook", "e-book", "downloads", "whitepaper"],
		"COURSES": ["course", "academy", "learning"],
		"RECENT_BLOG": ["blog", "insights", "articles"],
		"TESTIMONIALS": ["testimonial", "reviews", "case-study"],
		"WEBINAR": ["webinar", "event", "session"],
		"SERVICES": ["service", "solution", "capability"],
		"PODCAST": ["podcast", "listen", "episodes"],
		"SHOP": ["shop", "store", "buy"]
}

COLUMN_TO_READ_URL_FROM = "G"
COLUMN_TO_WRITE_URL_TO = {
		"ABOUT_US": "M",
		"EBOOK": "N",
		"COURSES": "O",
		"RECENT_BLOG": "P",
		"TESTIMONIALS": "Q",
		"WEBINAR": "R",
		"SERVICES": "S",
		"PODCAST": "T",
		"SHOP": "U"
}

CACHE_DIR = "firecrawl_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

CATEGORY_RULES = {
		'ABOUT_US': 'ascending',
		'EBOOK': 'ascending',
		'COURSES': 'ascending',
		'RECENT_BLOG': 'descending',
		'TESTIMONIALS': 'ascending',
		'WEBINAR': 'descending',
		'SERVICES': 'descending',
		'PODCAST': 'descending',
		'SHOP': 'ascending'
}

EXTRACTION_METADATA_COLUMN = "V"  # Column V for metadata

# Calculate URL Depth
def calculate_url_depth(url: str) -> int:
		try:
				parsed = urlparse(url)
				path = parsed.path.strip('/').split('/')
				return len(path)
		except Exception:
				return -1  # Invalid URL, will be skipped

# Deepest Point Function
def deepest_point_function(url_depth_pairs: List[Tuple[str, int]], category: str) -> List[str]:
		if category not in CATEGORY_RULES:
				raise ValueError(f"Category {category} not found in CATEGORY_RULES")
		
		sort_order = CATEGORY_RULES[category]
		if sort_order == 'ascending':
				top_urls = sorted(url_depth_pairs, key=lambda x: x[1])[:10]
		else:  # descending
				top_urls = sorted(url_depth_pairs, key=lambda x: x[1], reverse=True)[:10]
		
		return [url for url, _ in top_urls]

def truncate_to_bytes(text: str, max_bytes: int) -> str:
	encoded = text.encode('utf-8')
	if len(encoded) <= max_bytes:
			return text
	# Find a safe cut-off point
	truncated = encoded[:max_bytes]
	return truncated.decode('utf-8', errors='ignore')

def safe_join(contents: List[str], max_bytes: int = 50000, delimiter: str = " --- NEXT CONTENT FROM HERE --- ") -> str:
    final_text = ""
    for content in contents:
        candidate = final_text + (delimiter if final_text else "") + content
        if len(candidate.encode('utf-8')) > max_bytes:
            break
        final_text = candidate
    return final_text

# FirecrawlWrapper
class FirecrawlWrapper:
		def __init__(self, api_key):
				self.app = FirecrawlApp(api_key=api_key)

		def _hash_url(self, url: str) -> str:
				return hashlib.md5(url.encode()).hexdigest()

		def _get_cache_path(self, url: str) -> str:
				return os.path.join(CACHE_DIR, f"{self._hash_url(url)}.json")

		def map_url(self, url: str) -> List[str]:
				cache_path = self._get_cache_path(url)
				if os.path.exists(cache_path):
						with open(cache_path, 'r') as f:
								links = json.load(f)
								print(f"Loaded {len(links)} cached links for {url}")
								return links
				try:
						result = self.app.map_url(url)
						if getattr(result, 'success', False):
								links = result.links
								with open(cache_path, 'w') as f:
										json.dump(links, f, indent=2)
								print(f"Firecrawl found {len(links)} links for {url}")
								return links
						else:
								print(f"Firecrawl failed for {url}")
								return []
				except Exception as e:
						print(f"Firecrawl error for {url}: {e}")
						return []

		def filter_by_category(self, urls: List[str], category: str) -> List[str]:
				keywords = CATEGORY_KEYWORDS.get(category.upper(), [])
				if not keywords:
						print(f"No keywords defined for category {category}")
						return []
				return [u for u in urls if any(k in u.lower() for k in keywords)]

# extract_main_html_content
def extract_main_html_content(html: str) -> str:
		soup = BeautifulSoup(html, "html.parser")
		for tag in soup(["script", "style", "noscript"]):
				tag.decompose()
		main = soup.find("main") or soup.find("article")
		if main:
				return main.get_text(separator="\n", strip=True)
		candidates = [
				div for div in soup.find_all("div")
				if len(div.get_text(strip=True)) > 200
					 and not any(c in " ".join(div.get("class", [])).lower() for c in ["nav", "header", "footer", "popup"])
		]
		if candidates:
				return max(candidates, key=lambda d: len(d.get_text(strip=True))).get_text(separator="\n", strip=True)
		return soup.get_text(separator="\n", strip=True)

# crawl_and_select_content
async def crawl_and_select_content(urls: List[str]) -> List[str]:
		crawler_config = CrawlerRunConfig(
				deep_crawl_strategy=BFSDeepCrawlStrategy(max_depth=0),
				verbose=False
		)
		contents = []
		async with AsyncWebCrawler() as crawler:
				for url in urls:
						try:
								result = await asyncio.wait_for(crawler.arun(url, config=crawler_config), timeout=15)
								if result and result[0].html:
										text = extract_main_html_content(result[0].html)
										contents.append(text)
						except Exception as e:
								print(f"Error crawling {url}: {e}")
		return contents

# GoogleSheetsManager
class GoogleSheetsManager:
		def __init__(self, credentials_file: str):
				scopes = ['https://www.googleapis.com/auth/spreadsheets']
				creds = service_account.Credentials.from_service_account_file(credentials_file, scopes=scopes)
				self.service = build('sheets', 'v4', credentials=creds)

		def extract_spreadsheet_id(self, sheet_url: str) -> str:
				import re
				pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
				match = re.search(pattern, sheet_url)
				if match:
						return match.group(1)
				raise ValueError("Invalid Google Sheet URL")

		def get_urls(self, spreadsheet_id: str, start_row: int = 2) -> List[Dict]:
				range_name = f"{COLUMN_TO_READ_URL_FROM}{start_row}:{COLUMN_TO_READ_URL_FROM}"
				result = self.service.spreadsheets().values().get(spreadsheetId=spreadsheet_id, range=range_name).execute()
				values = result.get('values', [])
				return [(i + start_row, row[0]) for i, row in enumerate(values) if row and row[0].strip()]

		def update_result(self, spreadsheet_id: str, row: int, content: str, metadata: str, column_to_process: str):
				# Write content to COLUMN_TO_PROCESS
				content_range = f"{column_to_process}{row}"
				if len(content) > 50000:
						print(f"Truncating content from {len(content)} to 50000 characters.")
						content = truncate_to_bytes(content, 50000)
				self.service.spreadsheets().values().update(
						spreadsheetId=spreadsheet_id,
						range=content_range,
						valueInputOption='RAW',
						body={'values': [[content]]}
				).execute()
				print(f"Updated row {row} in column {column_to_process} with content")

				# Read existing metadata from EXTRACTION_METADATA_COLUMN
				metadata_range = f"{EXTRACTION_METADATA_COLUMN}{row}"
				try:
						existing_metadata = self.service.spreadsheets().values().get(
								spreadsheetId=spreadsheet_id,
								range=metadata_range
						).execute().get('values', [['']])[0][0]
				except Exception as e:
						print(f"Error reading existing metadata for row {row}: {e}")
						existing_metadata = ''

				# Parse existing metadata and update or append new metadata
				category = metadata.split('=')[0]  # Extract category from new metadata (e.g., 'PODCAST')
				if existing_metadata:
						metadata_parts = existing_metadata.split(',')
						updated_parts = []
						category_found = False
						for part in metadata_parts:
								if part.startswith(category + '='):
										# Update existing category with new numerical value
										updated_parts.append(metadata)
										category_found = True
								else:
										updated_parts.append(part)
						if not category_found:
								# Append new metadata if category not found
								updated_parts.append(metadata)
						new_metadata = ','.join(updated_parts)
				else:
						new_metadata = metadata

				# Write updated metadata to EXTRACTION_METADATA_COLUMN
				self.service.spreadsheets().values().update(
						spreadsheetId=spreadsheet_id,
						range=metadata_range,
						valueInputOption='RAW',
						body={'values': [[new_metadata]]}
				).execute()
				print(f"Updated row {row} in column {EXTRACTION_METADATA_COLUMN} with metadata: {new_metadata}")

# Main Processing Function
async def process_all_rows_firecrawl(sheet_url: str, credentials_file: str, firecrawl_api_key: str, category: str):
		sheet_mgr = GoogleSheetsManager(credentials_file)
		spreadsheet_id = sheet_mgr.extract_spreadsheet_id(sheet_url)
		urls = sheet_mgr.get_urls(spreadsheet_id, start_row=301)

		firecrawl = FirecrawlWrapper(api_key=firecrawl_api_key)

		# Use COLUMN_TO_WRITE_URL_TO directly for COLUMN_TO_PROCESS
		column_to_process = COLUMN_TO_WRITE_URL_TO.get(category.upper())
		if not column_to_process:
				raise ValueError(f"No column defined for category {category}")

		for i, (row_num, main_url) in enumerate(urls):
				print(f"\nProcessing row {row_num}: {main_url}")
				sub_urls = firecrawl.map_url(main_url)
				await asyncio.sleep(6.5)

				filtered = firecrawl.filter_by_category(sub_urls, category)

				if not filtered:
						# Write "No URL found" to COLUMN_TO_PROCESS and "CATEGORY=0" to EXTRACTION_METADATA_COLUMN
						sheet_mgr.update_result(spreadsheet_id, row_num, "No URL found", f"{category.upper()}=0", column_to_process)
						continue

				# Calculate depths and create (url, depth) pairs
				url_depth_pairs = []
				for url in filtered:
						depth = calculate_url_depth(url)
						if depth != -1:
								url_depth_pairs.append((url, depth))

				if not url_depth_pairs:
						# No valid URLs after depth calculation
						sheet_mgr.update_result(spreadsheet_id, row_num, "No URL found", f"{category.upper()}=0", column_to_process)
						continue

				# Sort based on CATEGORY_RULES
				sort_order = CATEGORY_RULES.get(category.upper())
				if not sort_order:
						raise ValueError(f"No sorting rule defined for category {category}")

				reverse_sort = sort_order == 'descending'
				sorted_url_depth_pairs = sorted(url_depth_pairs, key=lambda x: x[1], reverse=reverse_sort)

				num_urls = len(sorted_url_depth_pairs)
				if num_urls <= 10:
						# Take all URLs
						selected_urls = [url for url, _ in sorted_url_depth_pairs]
				else:
						# Select top 10 based on deepest_point_function
						selected_urls = deepest_point_function(sorted_url_depth_pairs, category.upper())

				# Crawl selected URLs and collect content
				contents = await crawl_and_select_content(selected_urls)

				if not contents:
						content = "No meaningful content found"
						metadata = f"{category.upper()}=0"
				else:
						content = safe_join(contents)
						metadata = f"{category.upper()}={len(contents)}"

				# Write to Google Sheet
				sheet_mgr.update_result(spreadsheet_id, row_num, content, metadata, column_to_process)

In [2]:
FIRECRAWL_API="fc-29599096ac8b426dbf178180c53500ed"
CREDENTIALS_FILE = 'data/url-to-email-445616-cebe4868914f.json'
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1wDaFAe5ayIB8zSyjub9QQfsFtyQcHdbgOR4YZmeY8vQ/edit?gid=0#gid=0" 

In [3]:
await process_all_rows_firecrawl(
		sheet_url=GOOGLE_SHEET_URL,
		credentials_file=CREDENTIALS_FILE,
		firecrawl_api_key=FIRECRAWL_API,
		category="SHOP"
)


Processing row 301: https://tableneeds.com
Loaded 519 cached links for https://tableneeds.com


[INIT].... → Crawl4AI 0.6.3 

Updated row 301 in column U with content
Updated row 301 in column V with metadata: TESTIMONIALS=2,COURSES=0,SERVICES=10,WEBINAR=2,PODCAST=0,EBOOK=0,RECENT_BLOG=10,ABOUT_US=6,SHOP=9

Processing row 302: https://metacomet.com
Loaded 67 cached links for https://metacomet.com
Updated row 302 in column U with content
Updated row 302 in column V with metadata: TESTIMONIALS=8,COURSES=2,SERVICES=9,WEBINAR=0,PODCAST=2,EBOOK=1,RECENT_BLOG=0,ABOUT_US=2,SHOP=0

Processing row 303: https://omnico.co
Loaded 35 cached links for https://omnico.co
Updated row 303 in column U with content
Updated row 303 in column V with metadata: TESTIMONIALS=0,COURSES=0,SERVICES=1,WEBINAR=0,PODCAST=0,EBOOK=0,RECENT_BLOG=1,ABOUT_US=0,SHOP=0

Processing row 304: https://bundygroup.com
Loaded 295 cached links for https://bundygroup.com


[INIT].... → Crawl4AI 0.6.3 

Updated row 304 in column U with content
Updated row 304 in column V with metadata: TESTIMONIALS=0,COURSES=1,SERVICES=10,WEBINAR=3,PODCAST=2,EBOOK=0,RECENT_BLOG=10,ABOUT_US=5,SHOP=3

Processing row 305: https://brookshoughton.com
Loaded 19 cached links for https://brookshoughton.com
Updated row 305 in column U with content
Updated row 305 in column V with metadata: TESTIMONIALS=0,COURSES=0,SERVICES=1,WEBINAR=0,PODCAST=0,EBOOK=0,RECENT_BLOG=1,ABOUT_US=2,SHOP=0

Processing row 306: https://securaconsultants.com
Loaded 47 cached links for https://securaconsultants.com
Updated row 306 in column U with content
Updated row 306 in column V with metadata: TESTIMONIALS=0,COURSES=1,SERVICES=0,WEBINAR=1,PODCAST=0,EBOOK=3,RECENT_BLOG=0,ABOUT_US=0,SHOP=0

Processing row 307: https://elliottadvisorygroup.com
Loaded 11 cached links for https://elliottadvisorygroup.com
Updated row 307 in column U with content
Updated row 307 in column V with metadata: TESTIMONIALS=0,COURSES=0,SERVICES=1,WEBINAR=0,PODC

...

Updated row 396 in column U with content
Updated row 396 in column V with metadata: TESTIMONIALS=2,COURSES=1,SERVICES=6,WEBINAR=10,PODCAST=7,EBOOK=0,RECENT_BLOG=2,ABOUT_US=10,SHOP=10

Processing row 397: https://kerriganadvisors.com
Loaded 684 cached links for https://kerriganadvisors.com


[INIT].... → Crawl4AI 0.6.3 

Updated row 397 in column U with content
Updated row 397 in column V with metadata: TESTIMONIALS=2,COURSES=1,SERVICES=6,WEBINAR=10,PODCAST=7,EBOOK=0,RECENT_BLOG=2,ABOUT_US=10,SHOP=10

Processing row 398: https://pbcatl.com
Loaded 12 cached links for https://pbcatl.com
Updated row 398 in column U with content
Updated row 398 in column V with metadata: TESTIMONIALS=0,COURSES=0,SERVICES=0,WEBINAR=1,PODCAST=0,EBOOK=0,RECENT_BLOG=0,ABOUT_US=1,SHOP=0

Processing row 399: https://greaterdubuque.org
Loaded 301 cached links for https://greaterdubuque.org
Updated row 399 in column U with content
Updated row 399 in column V with metadata: TESTIMONIALS=0,COURSES=0,SERVICES=9,WEBINAR=2,PODCAST=0,EBOOK=0,RECENT_BLOG=0,ABOUT_US=10,SHOP=0

Processing row 400: https://lakecountryadvisors.com
Loaded 414 cached links for https://lakecountryadvisors.com


[INIT].... → Crawl4AI 0.6.3 

Updated row 400 in column U with content
Updated row 400 in column V with metadata: TESTIMONIALS=10,COURSES=1,SERVICES=10,WEBINAR=1,PODCAST=0,EBOOK=0,RECENT_BLOG=1,ABOUT_US=10,SHOP=10
